In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

X = np.load("data/X.npy")
y = np.load("data/y.npy")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("train:", X_train.shape, "active:", int(y_train.sum()))
print("test: ", X_test.shape, "active:", int(y_test.sum()))

train: (3231, 2048) active: 2410
test:  (808, 2048) active: 603


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

def scores(model, Xa, ya):
    p = model.predict(Xa)
    return (accuracy_score(ya, p),
            precision_score(ya, p, zero_division=0),
            recall_score(ya, p, zero_division=0),
            f1_score(ya, p, zero_division=0))

models = {
    "Naive Bayes": BernoulliNB(),
    "KNN": Pipeline([("sc", StandardScaler()),
                     ("m", KNeighborsClassifier(n_neighbors=5))]),
    "Decision Tree": DecisionTreeClassifier(random_state=42,
                                            class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=500, n_jobs=-1,
                                            random_state=42,
                                            class_weight="balanced"),
    "SVM": Pipeline([("sc", StandardScaler()),
                     ("m", SVC(kernel="rbf", class_weight="balanced",
                               random_state=42))]),
    "Logistic Regression": Pipeline([("sc", StandardScaler()),
                     ("m", LogisticRegression(max_iter=2000,
                                              class_weight="balanced"))]),
    "ANN": Pipeline([("sc", StandardScaler()),
                     ("m", MLPClassifier(hidden_layer_sizes=(256, 64),
                                         max_iter=300, early_stopping=True,
                                         random_state=42))]),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    tr_acc, _, _, _ = scores(model, X_train, y_train)
    te_acc, prec, rec, f1 = scores(model, X_test, y_test)
    try:
        s = (model.predict_proba(X_test)[:, 1]
             if hasattr(model, "predict_proba")
             else model.decision_function(X_test))
        auc = roc_auc_score(y_test, s)
    except Exception:
        auc = np.nan
    rows.append({"model": name, "train acc": tr_acc, "test acc": te_acc,
                 "precision": prec, "recall": rec, "f1": f1, "roc auc": auc})
    print(name, "done")

results = pd.DataFrame(rows).round(3)
results.to_csv("data/model_results.csv", index=False)
results

Naive Bayes done
KNN done
Decision Tree done
Random Forest done
SVM done
Logistic Regression done
ANN done


,model,train acc,test acc,precision,recall,f1,roc auc
0,Naive Bayes,0.869,0.855,0.937,0.864,0.899,0.904
1,KNN,0.920,0.880,0.878,0.975,0.924,0.889
2,Decision Tree,0.998,0.899,0.929,0.935,0.932,0.863
3,Random Forest,0.996,0.923,0.941,0.957,0.949,0.961
4,SVM,0.966,0.897,0.945,0.915,0.930,0.939
5,Logistic Regression,0.991,0.892,0.924,0.932,0.928,0.943
6,ANN,0.990,0.908,0.930,0.949,0.939,0.955


In [3]:
from collections import defaultdict
from rdkit.Chem.Scaffolds import MurckoScaffold

lab = pd.read_csv("data/tlr7_labelled.csv")
smiles = lab["smiles"].tolist()
assert len(smiles) == X.shape[0], "order mismatch"

groups = defaultdict(list)
for i, smi in enumerate(smiles):
    core = MurckoScaffold.MurckoScaffoldSmiles(smiles=smi, includeChirality=False)
    groups[core].append(i)

sets = sorted(groups.values(), key=len, reverse=True)
n_train = int(0.8 * len(smiles))

tr_idx, te_idx = [], []
for s in sets:
    (tr_idx if len(tr_idx) < n_train else te_idx).extend(s)

print("different cores:", len(sets))
print("train:", len(tr_idx), " test:", len(te_idx))
print("active share — train:", round(y[tr_idx].mean(), 3),
      " test:", round(y[te_idx].mean(), 3))

different cores: 1171
train: 3231  test: 808
active share — train: 0.786  test: 0.588


In [4]:
def evaluate(X_tr, y_tr, X_te, y_te):
    models = {
        "Naive Bayes": BernoulliNB(),
        "KNN": Pipeline([("sc", StandardScaler()),
                         ("m", KNeighborsClassifier(n_neighbors=5))]),
        "Decision Tree": DecisionTreeClassifier(random_state=42,
                                                class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=500, n_jobs=-1,
                                                random_state=42,
                                                class_weight="balanced"),
        "SVM": Pipeline([("sc", StandardScaler()),
                         ("m", SVC(kernel="rbf", class_weight="balanced",
                                   random_state=42))]),
        "Logistic Regression": Pipeline([("sc", StandardScaler()),
                         ("m", LogisticRegression(max_iter=2000,
                                                  class_weight="balanced"))]),
        "ANN": Pipeline([("sc", StandardScaler()),
                         ("m", MLPClassifier(hidden_layer_sizes=(256, 64),
                                             max_iter=300, early_stopping=True,
                                             random_state=42))]),
    }
    rows = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        tr_acc, _, _, _ = scores(model, X_tr, y_tr)
        te_acc, prec, rec, f1 = scores(model, X_te, y_te)
        try:
            s = (model.predict_proba(X_te)[:, 1]
                 if hasattr(model, "predict_proba")
                 else model.decision_function(X_te))
            auc = roc_auc_score(y_te, s)
        except Exception:
            auc = np.nan
        rows.append({"model": name, "train acc": tr_acc, "test acc": te_acc,
                     "precision": prec, "recall": rec, "f1": f1, "roc auc": auc})
        print(name, "done")
    return pd.DataFrame(rows).round(3)

scaffold = evaluate(X[tr_idx], y[tr_idx], X[te_idx], y[te_idx])
scaffold.to_csv("data/model_results_scaffold.csv", index=False)
scaffold

Naive Bayes done
KNN done
Decision Tree done
Random Forest done
SVM done
Logistic Regression done
ANN done


,model,train acc,test acc,precision,recall,f1,roc auc
0,Naive Bayes,0.876,0.823,0.937,0.749,0.833,0.902
1,KNN,0.926,0.750,0.718,0.945,0.816,0.815
2,Decision Tree,0.998,0.811,0.827,0.857,0.842,0.800
3,Random Forest,0.995,0.863,0.894,0.869,0.882,0.939
4,SVM,0.967,0.829,0.869,0.836,0.852,0.898
5,Logistic Regression,0.986,0.806,0.814,0.867,0.840,0.882
6,ANN,0.952,0.860,0.871,0.895,0.883,0.929


In [5]:
def evaluate(X_tr, y_tr, X_te, y_te):
    models = {
        "Naive Bayes": BernoulliNB(),
        "KNN": Pipeline([("sc", StandardScaler()),
                         ("m", KNeighborsClassifier(n_neighbors=5))]),
        "Decision Tree": DecisionTreeClassifier(random_state=42,
                                                class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=500, n_jobs=-1,
                                                random_state=42,
                                                class_weight="balanced"),
        "SVM": Pipeline([("sc", StandardScaler()),
                         ("m", SVC(kernel="rbf", class_weight="balanced",
                                   random_state=42))]),
        "Logistic Regression": Pipeline([("sc", StandardScaler()),
                         ("m", LogisticRegression(max_iter=2000,
                                                  class_weight="balanced"))]),
        "ANN": Pipeline([("sc", StandardScaler()),
                         ("m", MLPClassifier(hidden_layer_sizes=(256, 64),
                                             max_iter=300, early_stopping=True,
                                             random_state=42))]),
    }
    rows = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        tr_acc, _, _, _ = scores(model, X_tr, y_tr)
        te_acc, prec, rec, f1 = scores(model, X_te, y_te)
        try:
            s = (model.predict_proba(X_te)[:, 1]
                 if hasattr(model, "predict_proba")
                 else model.decision_function(X_te))
            auc = roc_auc_score(y_te, s)
        except Exception:
            auc = np.nan
        rows.append({"model": name, "train acc": tr_acc, "test acc": te_acc,
                     "precision": prec, "recall": rec, "f1": f1, "roc auc": auc})
        print(name, "done")
    return pd.DataFrame(rows).round(3)

scaffold = evaluate(X[tr_idx], y[tr_idx], X[te_idx], y[te_idx])
scaffold.to_csv("data/model_results_scaffold.csv", index=False)
scaffold

Naive Bayes done
KNN done
Decision Tree done
Random Forest done
SVM done
Logistic Regression done
ANN done


,model,train acc,test acc,precision,recall,f1,roc auc
0,Naive Bayes,0.876,0.823,0.937,0.749,0.833,0.902
1,KNN,0.926,0.750,0.718,0.945,0.816,0.815
2,Decision Tree,0.998,0.811,0.827,0.857,0.842,0.800
3,Random Forest,0.995,0.863,0.894,0.869,0.882,0.939
4,SVM,0.967,0.829,0.869,0.836,0.852,0.898
5,Logistic Regression,0.986,0.806,0.814,0.867,0.840,0.882
6,ANN,0.952,0.860,0.871,0.895,0.883,0.929


In [6]:
rand = pd.read_csv("data/model_results.csv")
cmp = pd.DataFrame({
    "model": rand["model"],
    "random test acc": rand["test acc"],
    "scaffold test acc": scaffold["test acc"],
    "drop": (rand["test acc"] - scaffold["test acc"]).round(3),
    "random AUC": rand["roc auc"],
    "scaffold AUC": scaffold["roc auc"],
})
cmp

,model,random test acc,scaffold test acc,drop,random AUC,scaffold AUC
0,Naive Bayes,0.855,0.823,0.032,0.904,0.902
1,KNN,0.880,0.750,0.130,0.889,0.815
2,Decision Tree,0.899,0.811,0.088,0.863,0.800
3,Random Forest,0.923,0.863,0.060,0.961,0.939
4,SVM,0.897,0.829,0.068,0.939,0.898
5,Logistic Regression,0.892,0.806,0.086,0.943,0.882
6,ANN,0.908,0.860,0.048,0.955,0.929


In [7]:
print("train:", len(tr_idx), " test:", len(te_idx))
print("active share — train:", round(y[tr_idx].mean(), 3),
      " test:", round(y[te_idx].mean(), 3))

train: 3231  test: 808
active share — train: 0.786  test: 0.588


In [8]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

Xtr, ytr = X[tr_idx], y[tr_idx]
Xte, yte = X[te_idx], y[te_idx]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    {"max_depth": [5, 10, 15, 20, None],
     "min_samples_leaf": [1, 5, 10]},
    cv=cv, scoring="roc_auc", n_jobs=-1)
dt.fit(Xtr, ytr)
print("Decision Tree best:", dt.best_params_, " CV AUC:", round(dt.best_score_, 3))

Decision Tree best: {'max_depth': None, 'min_samples_leaf': 10}  CV AUC: 0.919


In [9]:
rf = GridSearchCV(
    RandomForestClassifier(n_estimators=300, random_state=42,
                           class_weight="balanced", n_jobs=-1),
    {"max_depth": [None, 20],
     "max_features": ["sqrt", 0.1],
     "min_samples_leaf": [1, 3]},
    cv=cv, scoring="roc_auc", n_jobs=1)
rf.fit(Xtr, ytr)
print("Random Forest best:", rf.best_params_, " CV AUC:", round(rf.best_score_, 3))

Random Forest best: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1}  CV AUC: 0.959


In [10]:
for name, model in [("Decision Tree (tuned)", dt.best_estimator_),
                    ("Random Forest (tuned)", rf.best_estimator_)]:
    tr_acc, _, _, _ = scores(model, Xtr, ytr)
    te_acc, p, r, f = scores(model, Xte, yte)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    print(f"{name}\n  train {tr_acc:.3f} | test {te_acc:.3f} | "
          f"prec {p:.3f} | rec {r:.3f} | f1 {f:.3f} | auc {auc:.3f}")

Decision Tree (tuned)
  train 0.907 | test 0.827 | prec 0.914 | rec 0.779 | f1 0.841 | auc 0.890
Random Forest (tuned)
  train 0.996 | test 0.860 | prec 0.893 | rec 0.865 | f1 0.879 | auc 0.939


In [11]:
import pandas as pd

url = ("https://raw.githubusercontent.com/theochem/B3DB/main/"
       "B3DB/B3DB_classification.tsv")
bbb = pd.read_csv(url, sep="\t")

print(bbb.shape)
print(bbb.columns.tolist())

(7807, 12)
['NO.', 'compound_name', 'IUPAC_name', 'SMILES', 'CID', 'logBB', 'BBB+/BBB-', 'Inchi', 'threshold', 'reference', 'group', 'comments']


In [12]:
import numpy as np
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Crippen
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

DESC = {
    "MolWt":      Descriptors.MolWt,          # size
    "LogP":       Crippen.MolLogP,            # greasiness
    "TPSA":       Descriptors.TPSA,           # polar surface area
    "HBD":        Descriptors.NumHDonors,
    "HBA":        Descriptors.NumHAcceptors,
    "RotB":       Descriptors.NumRotatableBonds,
    "Rings":      Descriptors.RingCount,
    "AromRings":  Descriptors.NumAromaticRings,
    "FracCSP3":   Descriptors.FractionCSP3,
    "HeavyAtoms": Descriptors.HeavyAtomCount,
    "MolMR":      Crippen.MolMR,
    "NOCount":    Descriptors.NOCount,
}

def describe(smiles_list):
    rows, keep = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        rows.append([f(mol) for f in DESC.values()])
        keep.append(i)
    return pd.DataFrame(rows, columns=list(DESC)), keep

bbb = bbb.dropna(subset=["SMILES", "BBB+/BBB-"])
bbb_smiles = bbb["SMILES"].tolist()
Xb_df, keep = describe(bbb_smiles)
yb = (bbb["BBB+/BBB-"].values[keep] == "BBB+").astype(int)
bbb_smiles = [bbb_smiles[i] for i in keep]

print("molecules:", Xb_df.shape)
print("BBB+ :", int(yb.sum()), "   BBB- :", int((yb == 0).sum()))
Xb_df.describe().round(1)

molecules: (7805, 12)
BBB+ : 4956    BBB- : 2849


,MolWt,LogP,TPSA,HBD,HBA,RotB,Rings,AromRings,FracCSP3,HeavyAtoms,MolMR,NOCount
count,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0,7805.0
mean,385.5,2.4,87.0,2.0,5.3,4.8,3.3,1.3,0.5,26.9,101.5,6.2
std,170.9,2.2,67.7,2.3,3.7,3.6,1.6,1.1,0.2,12.0,43.1,4.3
min,6.9,-8.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
25%,285.3,1.2,41.5,1.0,3.0,2.0,2.0,0.0,0.3,20.0,78.1,3.0
50%,362.5,2.5,72.7,1.0,4.0,4.0,3.0,1.0,0.5,26.0,97.3,5.0
75%,451.6,3.8,107.6,3.0,6.0,6.0,4.0,2.0,0.7,31.0,116.6,8.0
max,1882.3,12.6,662.4,24.0,33.0,53.0,16.0,8.0,1.0,136.0,517.7,42.0
